In [2]:
import kagglehub
import os
import shutil
import pandas as pd

c:\Users\winni\Documents\Project\ocular_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# 1. Define your local project target paths
PROJECT_DATA_DIR = "data"
LOCAL_CSV_PATH = os.path.join(PROJECT_DATA_DIR, "full_df.csv")

# 2. Create the 'data' folder inside your project if it doesn't exist
if not os.path.exists(PROJECT_DATA_DIR):
    os.makedirs(PROJECT_DATA_DIR)
    print(f"Created local directory: {PROJECT_DATA_DIR}")

# 3. Only download and copy if the file isn't already in your project
if not os.path.exists(LOCAL_CSV_PATH):
    print("Fetching file from cache...")
    cache_dir = kagglehub.dataset_download("andrewmvd/ocular-disease-recognition-odir5k")
    cache_csv_path = os.path.join(cache_dir, "full_df.csv")
    
    # Copy the file to your project folder
    shutil.copy(cache_csv_path, LOCAL_CSV_PATH)
    print(f"Successfully copied full_df.csv to local path: {LOCAL_CSV_PATH}")
else:
    print(f"File already exists locally at: {LOCAL_CSV_PATH}")

# 4. Clean load using your new local project path
df = pd.read_csv(
    LOCAL_CSV_PATH, 
    encoding="utf-8", 
    encoding_errors="replace",
    engine="python",
    on_bad_lines="skip"
)

print(f"\nLoaded {len(df)} rows from your project folder.")


File already exists locally at: data\full_df.csv

Loaded 6392 rows from your project folder.


In [4]:
path = './data/full_df.csv'

df = pd.read_csv(path)
print(df.shape)
print(df.head(5))


(6392, 19)
   ID  Patient Age Patient Sex Left-Fundus Right-Fundus  \
0   0           69      Female  0_left.jpg  0_right.jpg   
1   1           57        Male  1_left.jpg  1_right.jpg   
2   2           42        Male  2_left.jpg  2_right.jpg   
3   4           53        Male  4_left.jpg  4_right.jpg   
4   5           50      Female  5_left.jpg  5_right.jpg   

                            Left-Diagnostic Keywords  \
0                                           cataract   
1                                      normal fundus   
2  laser spot，moderate non proliferative retinopathy   
3                        macular epiretinal membrane   
4             moderate non proliferative retinopathy   

                Right-Diagnostic Keywords  N  D  G  C  A  H  M  O  \
0                           normal fundus  0  0  0  1  0  0  0  0   
1                           normal fundus  1  0  0  0  0  0  0  0   
2  moderate non proliferative retinopathy  0  1  0  0  0  0  0  1   
3       mild nonproli

In [5]:
df_glaucoma = df[df['G'] == 1]
print(df_glaucoma.shape)
print(df_glaucoma.head())
print(df_glaucoma['N'].unique())

(397, 19)
      ID  Patient Age Patient Sex   Left-Fundus   Right-Fundus  \
34    43           35        Male   43_left.jpg   43_right.jpg   
78    95           46        Male   95_left.jpg   95_right.jpg   
129  153           79        Male  153_left.jpg  153_right.jpg   
141  167           71        Male  167_left.jpg  167_right.jpg   
150  178           54        Male  178_left.jpg  178_right.jpg   

                          Left-Diagnostic Keywords  \
34            wet age-related macular degeneration   
78                              suspected glaucoma   
129                                       glaucoma   
141                                       glaucoma   
150  dry age-related macular degeneration，glaucoma   

                         Right-Diagnostic Keywords  N  D  G  C  A  H  M  O  \
34   dry age-related macular degeneration，glaucoma  0  0  1  0  1  0  0  0   
78                        hypertensive retinopathy  0  0  1  0  0  1  0  0   
129           dry age-related macu

In [6]:
df_normal = df[df['N'] == 1]
print(df_normal.shape)
print(df_normal.head())

df_normal['target'].unique()

(2101, 19)
      ID  Patient Age Patient Sex   Left-Fundus   Right-Fundus  \
1      1           57        Male    1_left.jpg    1_right.jpg   
7      8           59        Male    8_left.jpg    8_right.jpg   
68    84           51      Female   84_left.jpg   84_right.jpg   
163  191           51      Female  191_left.jpg  191_right.jpg   
344  394           63        Male  394_left.jpg  394_right.jpg   

    Left-Diagnostic Keywords Right-Diagnostic Keywords  N  D  G  C  A  H  M  \
1              normal fundus             normal fundus  1  0  0  0  0  0  0   
7              normal fundus             normal fundus  1  0  0  0  0  0  0   
68             normal fundus             normal fundus  1  0  0  0  0  0  0   
163            normal fundus             normal fundus  1  0  0  0  0  0  0   
344            normal fundus             normal fundus  1  0  0  0  0  0  0   

     O                                           filepath labels  \
1    0  ../input/ocular-disease-recognition-odir5

<StringArray>
['[1, 0, 0, 0, 0, 0, 0, 0]']
Length: 1, dtype: str

In [7]:
filtered_df = pd.concat([df_normal, df_glaucoma])

print(filtered_df.shape)

# creating an array of labels using the glaucoma column
    # 0 - normal
    # 1 - glaucoma
y_labels = filtered_df['G']

print(y_labels.shape)

(2498, 19)
(2498,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(filtered_df, y_labels, test_size=0.2, random_state=123)

In [16]:
X_train.head()

,ID,Patient Age,Patient Sex,Left-Fundus,Right-Fundus,Left-Diagnostic Keywords,Right-Diagnostic Keywords,N,D,G,C,A,H,M,O,filepath,labels,target,filename
4933,2574,54,Female,2574_left.jpg,2574_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",2574_left.jpg
5457,3149,56,Male,3149_left.jpg,3149_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",3149_left.jpg
1600,2421,51,Female,2421_left.jpg,2421_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",2421_right.jpg
1960,2818,49,Male,2818_left.jpg,2818_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",2818_right.jpg
5379,3068,58,Male,3068_left.jpg,3068_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",3068_left.jpg
